# Measurement

Adapted from Qualtran v0.7.0, commit bcb17bb25360ed68ce3bc084645db3e929799389. Copyright Google LLC. SPDX-License-Identifier: Apache-2.0.

In [ ]:
from collections import Counter

import torch
from IPython.display import Markdown, SVG, display

from examples.algorithms.binary_field_arithmetic import FieldOperation
from examples.algorithms.field_addition_catalog import field_signature_svg
from flagquantum.compiler._component_trace.call_graph import symbolic_call_graph
from flagquantum.compiler._component_trace.qec_gate_cost import query_qec_gate_cost
from flagquantum.compiler._component_trace.resource_generalization import MeasurementCall
from flagquantum.compiler.registers.primitives.measurement import measure_x_clear_ir, measure_z_clear_ir
from flagquantum.core.ir.program.register_types import classical_bit_type, quantum_bit_type
from flagquantum.core.ir.program.registers import Register, RegisterSide
from flagquantum.core.ir.program.signatures import Signature
from flagquantum.dynamic import DynamicCircuit
from flagquantum.drawer.call_graph import export_call_graph_dot, format_call_graph_markdown, render_call_graph_svg
from flagquantum.runtime.dynamic.execution import run_dynamic

## `MeasureZ`
Measure a qubit in the Z basis.

#### Registers
 - `q [LEFT]`: The qubit to measure.
 - `c [RIGHT]`: The classical measurement result.


In [ ]:
signature = Signature((
    Register("q", quantum_bit_type(), side=RegisterSide.LEFT),
    Register("c", classical_bit_type(), side=RegisterSide.RIGHT),
))

def show_model(name, key):
    graph = symbolic_call_graph(key, {}, max_depth=1)
    assert graph.leaf_counts == {key: 1} and not graph.direct_calls
    title = lambda _: name
    display(SVG(render_call_graph_svg(export_call_graph_dot(key, graph.direct_calls, title=title))))
    display(Markdown(format_call_graph_markdown(key, graph.direct_calls, graph.leaf_counts, title=title)))
    print(query_qec_gate_cost(key))

### Example Instances

In [ ]:
meas_z = MeasurementCall("z")

#### Graphical Signature

In [ ]:
display(SVG(field_signature_svg([("MeasureZ", signature, {})])))

### Call Graph

In [ ]:
show_model("MeasureZ", meas_z)

## `MeasureX`
Measure a qubit in the X basis.

#### Registers
 - `q [LEFT]`: The qubit to measure.
 - `c [RIGHT]`: The classical measurement result.


In [ ]:
emit_x = measure_x_clear_ir

### Example Instances

In [ ]:
meas_x = MeasurementCall("x")

#### Graphical Signature

In [ ]:
display(SVG(field_signature_svg([("MeasureX", signature, {})])))

### Call Graph

In [ ]:
show_model("MeasureX", meas_x)

### Append and inspect both outcomes

The logical measurement consumes `q` and returns `c`. A fixed-width dynamic circuit retains the physical slot and returns it to zero with feedback from the same classical result. X measurement first rotates the input with H. Both source graphs declare one measurement. The physical slot cleanup and basis rotation are counted separately in the emitted stream.

Use the same entangled input for both bases. Project it onto the basis bras to obtain each unnormalized survivor and its probability. Verify the normalized complex trajectory states against the corresponding branch, including the cleared physical wire.

In [ ]:
dtype = torch.complex128
state = torch.tensor([1, 0, 0, 1j], dtype=dtype) / 2**0.5
z_bras = torch.eye(2, dtype=dtype)
x_bras = torch.tensor([[1, 1], [1, -1]], dtype=dtype) / 2**0.5

for name, emit, bras in (("Z", measure_z_clear_ir, z_bras), ("X", emit_x, x_bras)):
    survivors = bras @ state.reshape(2, 2)
    probabilities = survivors.abs().square().sum(dim=1)
    torch.testing.assert_close(probabilities, torch.full((2,), 0.5, dtype=torch.float64))
    expected = torch.zeros((2, 4), dtype=dtype)
    expected[:, :2] = survivors / probabilities.sqrt()[:, None]
    op = FieldOperation(2, emit(0, 0))
    circuit = DynamicCircuit(2, inputs=state, dtype=dtype)
    circuit.append(op)
    result = run_dynamic(circuit, shots=32, seed=19, strategy="trajectory")
    outcomes = result.classical_bits[:, 0].to(torch.long)
    assert set(outcomes.tolist()) == {0, 1}
    torch.testing.assert_close(result.final_states.reshape(-1, 4), expected[outcomes], rtol=0, atol=1e-12)
    print(name, "probabilities", probabilities.tolist())
    print(name, "emitted", dict(Counter(gate.name for gate in op.instructions)))